# 🧙 O Mago Mestre — Gerador de Imagens Consistentes

Gera novas imagens do personagem **O Mago Mestre** em diferentes poses e situações,
mantendo fidelidade total ao DNA visual do personagem usando imagens de referência.

### Estratégia de modelos (3 camadas)
| Camada | Modelo | Requisito | Melhor para |
|--------|--------|-----------|-------------|
| **1** | Gemini 2.5 Flash Image | Google AI Studio API key (grátis) | Consistência com referências visuais |
| **2** | Imagen 3 | Vertex AI / GCP (pago) | Alta qualidade artística |
| **3** | Flux Dev + LoRA | GPU A100 no Colab (pago) | Consistência absoluta pós-treino |

> **Recomendado:** Camada 1 (Gemini) — suporta até 14 imagens de referência, melhor custo-benefício.

---

### Como usar
1. Execute **Célula 1** (Setup) para instalar dependências
2. Configure sua **API Key** em `Secrets` (ícone de cadeado no painel esquerdo)
3. Execute **Célula 2** para carregar as imagens de referência
4. Use a **Célula 5** para gerar novas imagens
5. Use a **Célula 6** para geração em lote

## ⚙️ Célula 1 — Setup e Dependências

In [ ]:
# Instalar dependências
!pip install -q google-genai pillow tqdm requests ipywidgets

import os
import io
import json
import base64
import random
import zipfile
import textwrap
from pathlib import Path
from io import BytesIO
from datetime import datetime
from IPython.display import display, Image as IPyImage, HTML, clear_output

import PIL.Image
import requests
from tqdm.notebook import tqdm

# Detectar ambiente (Colab vs Jupyter local)
try:
    from google.colab import userdata, files, drive
    IS_COLAB = True
    print("✅ Ambiente: Google Colab")
except ImportError:
    IS_COLAB = False
    print("ℹ️ Ambiente: Jupyter local")

# Configurar API Key
# No Colab: adicione GOOGLE_API_KEY em Secrets (ícone 🔑 no painel esquerdo)
# Localmente: defina a variável de ambiente antes de executar
if IS_COLAB:
    try:
        GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
        print("✅ Google API Key carregada dos Secrets")
    except Exception:
        GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
        print("⚠️ API Key não encontrada nos Secrets. Configure em Secrets > GOOGLE_API_KEY")
else:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
    if not GOOGLE_API_KEY:
        print("⚠️ Configure: export GOOGLE_API_KEY='sua-chave'")
    else:
        print("✅ Google API Key carregada da variável de ambiente")

print(f"\n🔑 API Key: {'✓ configurada' if GOOGLE_API_KEY else '✗ ausente — configure antes de continuar'}")

## 📁 Célula 2 — Carregar Imagens de Referência

Três opções disponíveis:
- **A)** Upload manual (qualquer ambiente)
- **B)** Do Google Drive (recomendado para Colab)
- **C)** Do projeto local (se executando em Jupyter local)

In [ ]:
# ============================================================
# CONFIGURAÇÃO — escolha a fonte das imagens de referência
# ============================================================

# Opção C: Caminho local do projeto (ajuste se necessário)
LOCAL_MAGO_DIR = Path("assets/backgrounds/mago")  # relativo ao notebook

# Opção B: Caminho no Google Drive (montar abaixo se necessário)
DRIVE_MAGO_DIR = Path("/content/drive/MyDrive/clip-flow/assets/backgrounds/mago")

# Diretório de saída das imagens geradas
OUTPUT_DIR = Path("generated_mago")
OUTPUT_DIR.mkdir(exist_ok=True)

# ============================================================

def montar_drive():
    """Monta o Google Drive (apenas Colab)."""
    if IS_COLAB:
        drive.mount('/content/drive')
        print("✅ Google Drive montado em /content/drive")
    else:
        print("ℹ️ Google Drive: apenas disponível no Colab")


def carregar_referencias_local(mago_dir: Path) -> dict[str, PIL.Image.Image]:
    """Carrega imagens de referência do diretório local."""
    referencias = {}
    extensoes = [".png", ".jpg", ".jpeg", ".webp"]
    
    if not mago_dir.exists():
        print(f"⚠️ Diretório não encontrado: {mago_dir}")
        return referencias
    
    for ext in extensoes:
        for img_path in sorted(mago_dir.glob(f"*{ext}")):
            try:
                img = PIL.Image.open(img_path).convert("RGB")
                referencias[img_path.name] = img
            except Exception as e:
                print(f"⚠️ Erro ao carregar {img_path.name}: {e}")
    
    return referencias


def fazer_upload_manual() -> dict[str, PIL.Image.Image]:
    """Upload manual de imagens via widget do Colab/Jupyter."""
    referencias = {}
    if IS_COLAB:
        print("📂 Selecione suas imagens de referência (múltiplos arquivos)...")
        uploaded = files.upload()
        for nome, dados in uploaded.items():
            try:
                img = PIL.Image.open(BytesIO(dados)).convert("RGB")
                referencias[nome] = img
                print(f"  ✓ {nome} ({img.size[0]}x{img.size[1]})")
            except Exception as e:
                print(f"  ✗ {nome}: {e}")
    else:
        print("ℹ️ Upload manual disponível apenas no Colab. Use carregar_referencias_local()")
    return referencias


def redimensionar_para_referencia(img: PIL.Image.Image, max_size: int = 1024) -> PIL.Image.Image:
    """Redimensiona mantendo aspect ratio, limitando ao tamanho máximo."""
    w, h = img.size
    if max(w, h) <= max_size:
        return img
    ratio = max_size / max(w, h)
    new_size = (int(w * ratio), int(h * ratio))
    return img.resize(new_size, PIL.Image.LANCZOS)


def selecionar_melhores_referencias(
    referencias: dict[str, PIL.Image.Image],
    n: int = 5,
    prioridade: list[str] | None = None
) -> list[PIL.Image.Image]:
    """
    Seleciona N imagens de referência mais diversas.
    Prioriza imagens por nome se especificado.
    """
    # Ordem de prioridade padrão: cobre poses variadas
    ORDEM_PADRAO = [
        "grey_wizard_front_pose_1",       # frontal (âncora)
        "grey_wizard_portrait_1",          # close-up
        "grey_wizard_staff_raised_1",      # cajado levantado
        "grey_wizard_spell_gesture_1",     # gesto de feitiço
        "mago_mestre_meditando_1",         # sentado/meditando
        "mago_mestre_tomando_cafe_1",      # com objeto
        "mago_mestre_lendo_grimorio_1",    # concentrado
        "mago_mestre_invocando_criatura_1",# ação dramática
        "mago_mestre_cozinhando_pocao_1", # corpo inteiro
        "grey_wizard_casting_magic_1",     # magia
    ]
    
    if prioridade:
        ordem = prioridade + [x for x in ORDEM_PADRAO if x not in prioridade]
    else:
        ordem = ORDEM_PADRAO
    
    selecionadas = []
    # Primeiro: imagens prioritárias
    for nome_base in ordem:
        if len(selecionadas) >= n:
            break
        for nome_arquivo, img in referencias.items():
            stem = Path(nome_arquivo).stem
            if nome_base in stem and img not in selecionadas:
                selecionadas.append(img)
                break
    
    # Completar com imagens restantes aleatórias
    restantes = [img for nome, img in referencias.items()
                 if img not in selecionadas]
    random.shuffle(restantes)
    selecionadas.extend(restantes[:n - len(selecionadas)])
    
    return selecionadas[:n]


def preview_referencias(referencias: dict[str, PIL.Image.Image], cols: int = 5):
    """Exibe um grid das imagens de referência carregadas."""
    if not referencias:
        print("Nenhuma imagem carregada.")
        return
    
    THUMB = 200
    nomes = list(referencias.keys())
    n = len(nomes)
    rows = (n + cols - 1) // cols
    
    grid_w = cols * (THUMB + 4)
    grid_h = rows * (THUMB + 24)
    grid = PIL.Image.new("RGB", (grid_w, grid_h), (30, 30, 40))
    
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(grid)
    
    for i, nome in enumerate(nomes):
        row, col = divmod(i, cols)
        img = referencias[nome].copy()
        img.thumbnail((THUMB, THUMB), PIL.Image.LANCZOS)
        x = col * (THUMB + 4) + 2
        y = row * (THUMB + 24) + 2
        grid.paste(img, (x, y))
        label = Path(nome).stem[:20]
        draw.text((x, y + THUMB + 2), label, fill=(200, 200, 200))
    
    display(grid)
    print(f"\n✅ {n} imagens de referência carregadas.")


# ============================================================
# CARREGAR IMAGENS
# ============================================================

# Tenta carregamento automático do projeto local
REFERENCIAS = {}

if LOCAL_MAGO_DIR.exists():
    print(f"📂 Carregando do projeto local: {LOCAL_MAGO_DIR}")
    REFERENCIAS = carregar_referencias_local(LOCAL_MAGO_DIR)
elif DRIVE_MAGO_DIR.exists():
    print(f"📂 Carregando do Google Drive: {DRIVE_MAGO_DIR}")
    REFERENCIAS = carregar_referencias_local(DRIVE_MAGO_DIR)
else:
    print("📂 Diretório local não encontrado.")
    print("   Opções:")
    print("   1. Monte o Drive: montar_drive() e verifique DRIVE_MAGO_DIR")
    print("   2. Faça upload: REFERENCIAS = fazer_upload_manual()")
    print("   3. Ajuste LOCAL_MAGO_DIR para o caminho correto")

if REFERENCIAS:
    print(f"\n📊 {len(REFERENCIAS)} imagens encontradas:")
    for nome in sorted(REFERENCIAS.keys()):
        img = REFERENCIAS[nome]
        print(f"   • {nome} ({img.size[0]}x{img.size[1]})")
    print("\n⬇️ Execute a próxima célula para ver o preview")

In [ ]:
# Preview das imagens de referência carregadas
if REFERENCIAS:
    preview_referencias(REFERENCIAS)
else:
    print("⚠️ Nenhuma imagem carregada ainda. Execute a célula anterior.")

## 🧬 Célula 3 — DNA do Personagem e Templates de Situações

In [ ]:
# ============================================================
# DNA DO PERSONAGEM — traços imutáveis em todo prompt
# ============================================================

CHARACTER_DNA = textwrap.dedent("""
    An elderly cartoon wizard with the following EXACT visual traits (NEVER change these):
    - Face: elongated face, prominent nose, thick white bushy eyebrows, weathered wise smile lines
    - Beard: very long white/silver wavy beard reaching his chest (NEVER short or straight)
    - Eyes: bright clear blue twinkling eyes, warm and expressive
    - Hat: tall pointed blue-grey hat with wide brim, slightly bent forward at tip (NEVER remove)
    - Robes: dark deep midnight blue flowing robes with subtle gold embroidery trim (NEVER white or brown)
    - Belt: brown leather belt with gold buckle at waist
    - Staff: gnarled twisted dark brown wooden staff with luminous golden glowing tip
    - Boots: worn brown leather boots
    - Style: semi-realistic cartoon, soft cel-shading, clean lines with subtle texture
    - Age: visibly elderly 70-80 years old (NEVER young)
    - Body: tall, lean, slightly hunched posture
    
    COLOR PALETTE (strict — do not deviate):
    Beard/Hair: #E8E8F0 (white-silver) | Hat: #4A5568 (blue-grey) |
    Robes: #1A1A3E (dark midnight blue) | Gold details: #C8A84E |
    Staff: #3E2723 (dark brown) | Staff glow: #FFD54F | Eyes: #64B5F6 (clear blue)
""").strip()

COMPOSITION = textwrap.dedent("""
    Vertical 4:5 aspect ratio (1080x1350 pixels).
    Character positioned in lower two-thirds of frame — lower third preferred.
    Upper 35-40% of image open and clear for text overlay.
    Soft depth of field on background elements.
    Dark atmospheric fantasy setting with warm golden accent lighting.
""").strip()

NEGATIVE_TRAITS = (
    "NOT photorealistic, NOT anime/manga style, NOT chibi, NOT young wizard, "
    "NOT threatening expression, NOT centered in frame, NOT bright white background, "
    "NOT wearing different colored robes, NOT without hat, NOT short beard"
)

# ============================================================
# TEMPLATES DE SITUAÇÕES (13 cenas + customizada)
# ============================================================

SITUACOES = {
    "sabedoria": {
        "label": "🧠 Sabedoria / Conselho",
        "acao": (
            "gentle knowing smile, one eyebrow raised wisely, leaning slightly on staff, "
            "free hand gesturing as if explaining something obvious"
        ),
        "cenario": "misty medieval forest with soft golden light filtering through ancient trees, dark moody atmosphere",
    },
    "confusao": {
        "label": "😕 Confuso / Perplexo",
        "acao": (
            "confused bewildered expression, head tilted to one side, "
            "one hand scratching long white beard, squinting eyes, floating question marks"
        ),
        "cenario": "dark medieval wizard study with floating books and magical scrolls, warm candlelight",
    },
    "segunda_feira": {
        "label": "😴 Segunda-feira / Cansado",
        "acao": (
            "extremely tired expression, heavy eyelids, hat drooping forward, "
            "holding enormous steaming coffee goblet with both hands, dark circles under eyes"
        ),
        "cenario": "early morning misty medieval kitchen, dawn light through stone window, cold blue atmosphere",
    },
    "vitoria": {
        "label": "🏆 Vitória / Celebração",
        "acao": (
            "triumphant joyful expression, staff raised high with golden magical particles erupting, "
            "hat slightly flying off, robes billowing dramatically, arms wide open"
        ),
        "cenario": "epic medieval castle courtyard at sunset, dramatic golden light, magical sparkles everywhere",
    },
    "tecnologia": {
        "label": "📱 Tecnologia / WiFi",
        "acao": (
            "surprised amused expression, holding glowing crystal ball like a smartphone, "
            "hat askew from leaning forward, floating digital glowing runes around device"
        ),
        "cenario": "anachronistic medieval office desk with candles and ancient books, blue glow from magical screen",
    },
    "cafe": {
        "label": "☕ Café / Manhã",
        "acao": (
            "content pleased expression, holding ornate ceramic mug with both hands, "
            "eyes slightly closed in pleasure, golden steam rising with magical sparkles"
        ),
        "cenario": "cozy medieval kitchen corner, warm fireplace glow, morning light",
    },
    "comida": {
        "label": "🍲 Comida / Culinária",
        "acao": (
            "focused proud expression, stirring large bubbling cauldron with wooden ladle, "
            "small apron over robes, floating magical ingredients, colorful vapors"
        ),
        "cenario": "magical medieval stone kitchen, warm firelight from below cauldron, herbs and potions on shelves",
    },
    "trabalho": {
        "label": "💼 Trabalho / Escritório",
        "acao": (
            "concentrated serious expression, sitting at wooden desk writing on parchment scroll, "
            "multiple scrolls and books open around, quill pen in hand"
        ),
        "cenario": "dark medieval scriptorium, warm golden candlelight on desk, bookshelves reaching ceiling",
    },
    "relaxando": {
        "label": "😌 Relaxando / Feriado",
        "acao": (
            "blissfully asleep expression, sitting in comfortable wooden chair, "
            "hat tipped over eyes, hands folded on belly, staff leaning against wall, soft snoring aura"
        ),
        "cenario": "cozy medieval tavern corner, warm fireplace, afternoon golden light, peaceful atmosphere",
    },
    "meditando": {
        "label": "🧘 Meditando / Zen",
        "acao": (
            "eyes closed, serene peaceful smile, sitting cross-legged on floating magical stone, "
            "staff hovering beside him, subtle ethereal blue-gold aura radiating outward"
        ),
        "cenario": "mystical mountain peak at twilight, northern lights aurora in purple-blue sky, floating clouds below",
    },
    "relacionamento": {
        "label": "💕 Relacionamento / Romance",
        "acao": (
            "gentle warm knowing smile, holding glowing pink heart-shaped crystal in both hands, "
            "eyes twinkling with warmth, soft magical pink and blue aura"
        ),
        "cenario": "moonlit enchanted garden with glowing flowers and fireflies, romantic soft atmosphere",
    },
    "confronto": {
        "label": "🚫 Confronto / 'Você Não Passa'",
        "acao": (
            "stern but comically determined expression, staff raised high with intense golden magical energy, "
            "one hand extended forward in firm STOP gesture, dramatic magical wind blowing robes"
        ),
        "cenario": "narrow ancient stone bridge over deep misty abyss, dramatic stormy sky, lightning in background",
    },
    "surpresa": {
        "label": "😲 Surpreso / Espantado",
        "acao": (
            "wide eyes of genuine shock, mouth open in surprise, both hands raised, "
            "hat blown slightly back by surprise, staff dropped sideways, exclamation sparks"
        ),
        "cenario": "dark medieval hall with dramatic revelation lighting, magical smoke and mirrors",
    },
}

print("✅ DNA do personagem e templates carregados!")
print(f"\n🎭 {len(SITUACOES)} situações pré-definidas disponíveis:")
for key, val in SITUACOES.items():
    print(f"   [{key:15s}] {val['label']}")
print("\n   + 'custom' — situação completamente livre")

## 🎨 Célula 4 — Configurar Cliente Gemini e Funções de Geração

In [ ]:
import time
from google import genai
from google.genai import types

# ============================================================
# CONFIGURAÇÃO DOS MODELOS
# ============================================================

# Modelos em ordem de preferência (tenta o primeiro, cai para o próximo se 404)
# Nano Banana      → gemini-2.5-flash-image           (estável, produção)
# Nano Banana 2    → gemini-2.0-flash-exp-image-generation (rápido)
# Nano Banana Pro  → gemini-3-pro-image-preview        (maior qualidade, quando disponível)
MODELOS_IMAGEM = [
    "gemini-2.5-flash-image",                  # Nano Banana — estável, produção
    "gemini-2.0-flash-exp-image-generation",   # Flash experimental com imagem
    "gemini-3.1-flash-image-preview",          # Nano Banana 2
    "gemini-3-pro-image-preview",              # Nano Banana Pro
]

# Parâmetros de geração
TEMPERATURA = 0.85          # criatividade (0.0–2.0)
N_REFERENCIAS = 5           # quantas imagens de referência enviar (3–7 recomendado)
RESOLUCAO_MAX_REF = 1024    # limitar tamanho das referências enviadas

# Retry para rate limit (429)
MAX_TENTATIVAS_429 = 2      # tentativas por modelo ao receber 429
ESPERA_BASE_429 = 60        # segundos de espera base para 429 (dobra a cada tentativa)

# ============================================================

if not GOOGLE_API_KEY:
    print("❌ GOOGLE_API_KEY não configurada! Volte à Célula 1.")
else:
    cliente = genai.Client(api_key=GOOGLE_API_KEY)
    print(f"✅ Cliente Gemini configurado")
    print(f"   Modelos (ordem de tentativa):")
    for m in MODELOS_IMAGEM:
        print(f"     • {m}")
    print(f"   Temperatura        : {TEMPERATURA}")
    print(f"   Referências/geração: {N_REFERENCIAS} imagens")
    print(f"   Retry 429          : {MAX_TENTATIVAS_429}x com espera de {ESPERA_BASE_429}s")


def construir_prompt_completo(
    situacao_key: str,
    descricao_custom: str = "",
    cenario_custom: str = "",
) -> str:
    if situacao_key == "custom":
        acao = descricao_custom or "standing pose holding staff, wise expression"
        cenario = cenario_custom or "dark moody medieval forest with golden atmospheric lighting"
    else:
        sit = SITUACOES.get(situacao_key, SITUACOES["sabedoria"])
        acao = sit["acao"]
        cenario = sit["cenario"]

    return f"""Generate an illustration of this wizard character matching the reference images EXACTLY.

CHARACTER (replicate precisely from reference):
{CHARACTER_DNA}

ACTION/POSE:
{acao}

SETTING/BACKGROUND:
{cenario}

COMPOSITION:
{COMPOSITION}

IMPORTANT — DO NOT:
{NEGATIVE_TRAITS}

The character must look IDENTICAL to the reference images in visual style, colors, and design.
Only the pose, expression, action, and background should change."""


def pil_para_part(img: PIL.Image.Image) -> types.Part:
    """Converte imagem PIL para Part do Gemini (inline bytes)."""
    img_r = redimensionar_para_referencia(img, RESOLUCAO_MAX_REF)
    buf = BytesIO()
    img_r.save(buf, format="JPEG", quality=90)
    return types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg")


def _tentar_gerar(cliente, modelo: str, partes: list, temperatura: float) -> PIL.Image.Image | None:
    """Tenta gerar imagem com um modelo. Retorna PIL.Image ou None."""
    response = cliente.models.generate_content(
        model=modelo,
        contents=partes,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE", "TEXT"],
            temperature=temperatura,
            image_config=types.ImageConfig(aspect_ratio="4:5"),
        ),
    )
    for part in response.candidates[0].content.parts:
        if hasattr(part, 'inline_data') and part.inline_data:
            return PIL.Image.open(BytesIO(part.inline_data.data))
        elif hasattr(part, 'text') and part.text:
            print(f"   📝 {part.text[:120]}")
    return None


def gerar_imagem(
    situacao_key: str = "sabedoria",
    descricao_custom: str = "",
    cenario_custom: str = "",
    n_referencias: int = N_REFERENCIAS,
    referencias_manual: list[PIL.Image.Image] | None = None,
    exibir: bool = True,
    salvar: bool = True,
    nome_arquivo: str = "",
) -> PIL.Image.Image | None:
    global REFERENCIAS

    if not GOOGLE_API_KEY:
        print("❌ API Key não configurada")
        return None

    if not REFERENCIAS and not referencias_manual:
        print("❌ Nenhuma imagem de referência carregada. Execute a Célula 2.")
        return None

    # Selecionar referências
    refs = referencias_manual[:n_referencias] if referencias_manual else \
           selecionar_melhores_referencias(REFERENCIAS, n=n_referencias)

    # Montar conteúdo: imagens (Part) + textos (str)
    prompt_texto = construir_prompt_completo(situacao_key, descricao_custom, cenario_custom)
    partes = []
    for img in refs:
        partes.append(pil_para_part(img))
    partes.append(
        "These are reference images of the character. "
        "Replicate the character's visual style EXACTLY in your generation."
    )
    partes.append(prompt_texto)

    label = SITUACOES.get(situacao_key, {}).get('label', situacao_key)
    print(f"🎨 Gerando: {label}")
    print(f"   Referências: {len(refs)} imagens")

    for modelo in MODELOS_IMAGEM:
        for tentativa in range(MAX_TENTATIVAS_429):
            try:
                imagem_gerada = _tentar_gerar(cliente, modelo, partes, TEMPERATURA)

                if imagem_gerada is None:
                    print(f"   ⚠️ {modelo}: resposta sem imagem")
                    break  # sem imagem nesta resposta — passa pro próximo modelo

                print(f"   ✅ {modelo} → {imagem_gerada.size[0]}x{imagem_gerada.size[1]}")

                if exibir:
                    display(imagem_gerada)

                if salvar:
                    if not nome_arquivo:
                        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                        nome_arquivo = f"mago_{situacao_key}_{ts}"
                    caminho = OUTPUT_DIR / f"{nome_arquivo}.png"
                    imagem_gerada.save(caminho, "PNG")
                    print(f"   💾 {caminho}")

                return imagem_gerada

            except Exception as e:
                msg = str(e)
                eh_rate_limit = "429" in msg or "RESOURCE_EXHAUSTED" in msg
                eh_nao_encontrado = "404" in msg or "NOT_FOUND" in msg

                if eh_nao_encontrado:
                    print(f"   ⚠️ {modelo}: modelo não disponível (404), tentando próximo...")
                    break  # próximo modelo imediatamente

                elif eh_rate_limit:
                    espera = ESPERA_BASE_429 * (2 ** tentativa)
                    if tentativa < MAX_TENTATIVAS_429 - 1:
                        print(f"   ⏳ {modelo}: cota esgotada (429). Aguardando {espera}s antes de tentar próximo modelo...")
                        time.sleep(espera)
                    else:
                        print(f"   ⚠️ {modelo}: cota persistentemente esgotada — tentando próximo modelo")
                    break  # passa pro próximo modelo após a espera

                else:
                    # Erro desconhecido
                    print(f"   ⚠️ {modelo}: {type(e).__name__}: {msg[:140]}")
                    break  # próximo modelo

    print("❌ Todos os modelos falharam.")
    print("💡 Se todos retornaram 429, a cota gratuita foi esgotada.")
    print("   Opções:")
    print("   1. Aguarde até amanhã (cota diária reseta à meia-noite UTC)")
    print("   2. Ative billing no Google AI Studio para cota maior")
    print("   3. Use uma API Key diferente com cota disponível")
    return None


print("\n✅ Pronto! Execute a Célula 5 para gerar.")


In [ ]:
# ============================================================
# DIAGNÓSTICO — descubra quais modelos estão disponíveis
# Execute esta célula para ver o nome correto dos modelos
# e ajuste MODELOS_IMAGEM na célula anterior se necessário
# ============================================================

print("🔍 Buscando modelos com suporte a geração de imagens...\n")

modelos_imagem_disponiveis = []
try:
    for model in cliente.models.list():
        nome = model.name
        # Filtra modelos com capacidade de geração de imagem
        tem_imagem = (
            "image" in nome.lower()
            or "imagen" in nome.lower()
            or (
                hasattr(model, 'supported_actions')
                and model.supported_actions
                and any("image" in str(a).lower() for a in model.supported_actions)
            )
        )
        if tem_imagem:
            modelos_imagem_disponiveis.append(nome)
            print(f"  ✓ {nome}")

    if not modelos_imagem_disponiveis:
        # Fallback: listar todos e deixar usuário escolher
        print("  (nenhum detectado automaticamente — listando todos:)")
        for model in cliente.models.list():
            print(f"  • {model.name}")

except Exception as e:
    print(f"⚠️ Erro ao listar modelos: {e}")

print("\n💡 Se os modelos em MODELOS_IMAGEM retornarem 404,")
print("   copie um nome desta lista e ajuste a variável MODELOS_IMAGEM na célula anterior.")


## 🎭 Célula 5 — Geração Individual

**Configure as variáveis abaixo e execute:**

In [ ]:
# ============================================================
# ✏️ CONFIGURE AQUI
# ============================================================

# Escolha uma situação pré-definida:
#   sabedoria | confusao | segunda_feira | vitoria | tecnologia |
#   cafe | comida | trabalho | relaxando | meditando |
#   relacionamento | confronto | surpresa | custom
SITUACAO = "segunda_feira"

# Se SITUACAO = 'custom', descreva ação e cenário em inglês:
ACAO_CUSTOM = "holding a glowing magical football, surprised excited expression, arms wide open"
CENARIO_CUSTOM = "mystical medieval stadium with magical floating torches, crowd of enchanted creatures"

# Quantas referências usar (3-7 recomendado, máximo 14)
N_REFS = 5

# ============================================================

# Mostrar situação selecionada
if SITUACAO == "custom":
    print(f"🎭 Situação: CUSTOM")
    print(f"   Ação   : {ACAO_CUSTOM}")
    print(f"   Cenário: {CENARIO_CUSTOM}")
else:
    sit = SITUACOES.get(SITUACAO)
    if sit:
        print(f"🎭 Situação: {sit['label']}")
        print(f"   Ação   : {sit['acao'][:80]}...")
        print(f"   Cenário: {sit['cenario'][:80]}...")
    else:
        print(f"⚠️ Situação '{SITUACAO}' não encontrada. Usando 'sabedoria'.")
        SITUACAO = "sabedoria"

print(f"   Referências: {N_REFS} imagens")
print()

# Gerar!
resultado = gerar_imagem(
    situacao_key=SITUACAO,
    descricao_custom=ACAO_CUSTOM,
    cenario_custom=CENARIO_CUSTOM,
    n_referencias=N_REFS,
    exibir=True,
    salvar=True,
)

## 📦 Célula 6 — Geração em Lote

Gera múltiplas imagens de uma vez, com barra de progresso.

In [ ]:
import time

# ============================================================
# ✏️ CONFIGURE O LOTE AQUI
# ============================================================

# Lista de situações a gerar (use chaves de SITUACOES ou dicts para custom)
LOTE = [
    "sabedoria",
    "segunda_feira",
    "cafe",
    "tecnologia",
    "confusao",
    "vitoria",
    "relaxando",
    "comida",
    {"key": "custom", "acao": "looking at a glowing magical mirror, pointing finger at reflection in disbelief",
     "cenario": "enchanted tower bedroom with magical floating candles"},
    {"key": "custom", "acao": "riding a flying broomstick with wind-blown robes, excited expression",
     "cenario": "night sky above medieval city rooftops, full moon background"},
]

N_REFS_LOTE = 4         # referências por geração (menos = mais rápido)
PAUSA_SEGUNDOS = 15     # pausa entre gerações (aumentar se receber 429 com frequência)
EXIBIR_DURANTE = True   # exibir cada imagem durante a geração

# ============================================================

print(f"📦 Gerando lote de {len(LOTE)} imagens...")
print(f"   Referências/imagem  : {N_REFS_LOTE}")
print(f"   Pausa entre gerações: {PAUSA_SEGUNDOS}s")
est_min = len(LOTE) * (15 + PAUSA_SEGUNDOS) // 60
est_max = len(LOTE) * (30 + PAUSA_SEGUNDOS) // 60
print(f"   Estimativa          : ~{est_min}–{est_max} min")
print(f"   💡 Cota gratuita: ~15 imagens/dia. Se receber 429, aguarde até amanhã ou ative billing.")
print()

resultados_lote = []
sucessos = 0
falhas = 0

for i, item in enumerate(tqdm(LOTE, desc="Gerando imagens")):
    print(f"\n{'='*50}")
    print(f"[{i+1}/{len(LOTE)}]")
    
    # Parsear item
    if isinstance(item, str):
        sit_key = item
        acao = ""
        cenario = ""
    elif isinstance(item, dict):
        sit_key = item.get("key", "custom")
        acao = item.get("acao", "")
        cenario = item.get("cenario", "")
    else:
        print(f"⚠️ Item inválido: {item}")
        falhas += 1
        continue
    
    # Nome de arquivo
    ts = datetime.now().strftime("%H%M%S")
    nome = f"lote_{i+1:02d}_{sit_key}_{ts}"
    
    # Gerar
    img = gerar_imagem(
        situacao_key=sit_key,
        descricao_custom=acao,
        cenario_custom=cenario,
        n_referencias=N_REFS_LOTE,
        exibir=EXIBIR_DURANTE,
        salvar=True,
        nome_arquivo=nome,
    )
    
    if img:
        resultados_lote.append({"situacao": sit_key, "imagem": img, "arquivo": nome})
        sucessos += 1
    else:
        falhas += 1
        # Falha pode indicar cota esgotada — pausa extra antes de continuar
        if falhas >= 2:
            print(f"\n⛔ {falhas} falhas consecutivas detectadas.")
            print("   Se o problema for 429, interrompa o lote e tente novamente amanhã.")
    
    # Pausa entre gerações (evitar rate limit)
    if i < len(LOTE) - 1:
        print(f"   ⏱️ Aguardando {PAUSA_SEGUNDOS}s antes da próxima geração...")
        time.sleep(PAUSA_SEGUNDOS)

print(f"\n{'='*50}")
print(f"✅ Lote concluído: {sucessos}/{len(LOTE)} geradas com sucesso")
if falhas:
    print(f"⚠️ {falhas} falhas — verifique se a cota foi esgotada")
print(f"📁 Imagens salvas em: {OUTPUT_DIR}/")


## 🔧 Célula 7 — Refinamento por Iteração (img2img)

Usa uma imagem gerada como nova referência para refinar fidelidade ou detalhes.

In [ ]:
# ============================================================
# ✏️ CONFIGURE O REFINAMENTO
# ============================================================

# Imagem base para refinar (None = usa a última gerada, ou passe PIL.Image diretamente)
IMAGEM_BASE = None  # ou: PIL.Image.open("generated_mago/mago_sabedoria_xxx.png")

# O que corrigir/melhorar
INSTRUCAO_REFINAMENTO = """
Keep this exact composition but improve:
- Make the beard longer and more flowing
- Ensure the hat color is dark blue-grey (#4A5568), not too bright
- Add more golden glow to the staff tip
- Make the robes darker (midnight blue #1A1A3E)
Keep everything else identical.
""".strip()

# ============================================================

def refinar_imagem(
    imagem_base: PIL.Image.Image,
    instrucao: str,
    referencias_adicionais: int = 3,
    exibir: bool = True,
    salvar: bool = True,
) -> PIL.Image.Image | None:
    """
    Refina uma imagem gerada usando ela mesma + referências originais.
    
    Args:
        imagem_base: imagem PIL a refinar
        instrucao: descrição do que melhorar (inglês)
        referencias_adicionais: quantas refs originais incluir
        exibir: exibir resultado no notebook
        salvar: salvar em OUTPUT_DIR
    """
    if not GOOGLE_API_KEY:
        print("❌ API Key não configurada")
        return None
    
    # Montar partes: imagem base + refs originais + instrução
    partes = []
    
    partes.append("THIS IS THE IMAGE TO REFINE:")
    partes.append(pil_para_part(imagem_base))
    
    # Referências originais
    if REFERENCIAS and referencias_adicionais > 0:
        refs = selecionar_melhores_referencias(REFERENCIAS, n=referencias_adicionais)
        partes.append(
            f"These {len(refs)} images are the ORIGINAL CHARACTER REFERENCES for consistency:"
        )
        for r in refs:
            partes.append(pil_para_part(r))
    
    # Instrução de refinamento
    prompt_refinamento = f"""{instrucao}

CHARACTER TRAITS TO ALWAYS MAINTAIN:
{CHARACTER_DNA}

COMPOSITION TO MAINTAIN:
{COMPOSITION}"""
    
    partes.append(prompt_refinamento)
    
    print(f"🔧 Refinando imagem...")
    print(f"   Base + {referencias_adicionais} referências originais")
    
    for modelo in MODELOS_IMAGEM:
        for tentativa in range(MAX_TENTATIVAS_429):
            try:
                response = cliente.models.generate_content(
                    model=modelo,
                    contents=partes,
                    config=types.GenerateContentConfig(
                        response_modalities=["IMAGE", "TEXT"],
                        temperature=0.5,  # menos criatividade para refinamento
                        image_config=types.ImageConfig(aspect_ratio="4:5"),
                    ),
                )
                
                imagem_refinada = None
                for part in response.candidates[0].content.parts:
                    if hasattr(part, 'inline_data') and part.inline_data:
                        imagem_refinada = PIL.Image.open(BytesIO(part.inline_data.data))
                        break
                
                if imagem_refinada is None:
                    print(f"   ⚠️ {modelo}: sem imagem na resposta")
                    break

                print(f"   ✅ {modelo} — Refinamento concluído ({imagem_refinada.size[0]}x{imagem_refinada.size[1]})")
                
                if exibir:
                    print("   Original → Refinado:")
                    w = 512
                    orig_t = imagem_base.copy()
                    ref_t = imagem_refinada.copy()
                    orig_t.thumbnail((w, w))
                    ref_t.thumbnail((w, w))
                    comparacao = PIL.Image.new(
                        "RGB",
                        (orig_t.width + ref_t.width + 10, max(orig_t.height, ref_t.height)),
                        (30, 30, 40)
                    )
                    comparacao.paste(orig_t, (0, 0))
                    comparacao.paste(ref_t, (orig_t.width + 10, 0))
                    display(comparacao)
                    display(imagem_refinada)
                
                if salvar:
                    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                    caminho = OUTPUT_DIR / f"mago_refinado_{ts}.png"
                    imagem_refinada.save(caminho, "PNG")
                    print(f"   💾 Salvo: {caminho}")
                
                return imagem_refinada
            
            except Exception as e:
                msg = str(e)
                eh_rate_limit = "429" in msg or "RESOURCE_EXHAUSTED" in msg
                eh_nao_encontrado = "404" in msg or "NOT_FOUND" in msg

                if eh_nao_encontrado:
                    print(f"   ⚠️ {modelo}: não disponível (404), tentando próximo...")
                    break
                elif eh_rate_limit:
                    espera = ESPERA_BASE_429 * (2 ** tentativa)
                    if tentativa < MAX_TENTATIVAS_429 - 1:
                        print(f"   ⏳ {modelo}: 429. Aguardando {espera}s...")
                        time.sleep(espera)
                    else:
                        print(f"   ⚠️ {modelo}: cota esgotada — tentando próximo")
                    break
                else:
                    print(f"   ⚠️ {modelo}: {type(e).__name__}: {msg[:140]}")
                    break

    print("❌ Refinamento falhou em todos os modelos.")
    return None


# Usar última imagem gerada ou a configurada acima
imagem_para_refinar = IMAGEM_BASE or resultado

if imagem_para_refinar is not None:
    imagem_refinada = refinar_imagem(
        imagem_base=imagem_para_refinar,
        instrucao=INSTRUCAO_REFINAMENTO,
        referencias_adicionais=3,
        exibir=True,
        salvar=True,
    )
else:
    print("ℹ️ Execute a Célula 5 primeiro para gerar uma imagem base, ou configure IMAGEM_BASE.")


## 📊 Célula 8 — Preview das Imagens Geradas

In [ ]:
def preview_geradas(diretorio: Path = OUTPUT_DIR, cols: int = 4):
    """Exibe grid com todas as imagens geradas no diretório."""
    imagens = sorted(list(diretorio.glob("*.png")) + list(diretorio.glob("*.jpg")))
    
    if not imagens:
        print(f"📂 Nenhuma imagem em {diretorio}")
        return
    
    print(f"📊 {len(imagens)} imagens geradas em {diretorio}/")
    print()
    
    THUMB = 250
    rows = (len(imagens) + cols - 1) // cols
    
    grid_w = cols * (THUMB + 4)
    grid_h = rows * (THUMB + 28)
    grid = PIL.Image.new("RGB", (grid_w, grid_h), (20, 20, 30))
    
    from PIL import ImageDraw
    draw = ImageDraw.Draw(grid)
    
    for i, img_path in enumerate(imagens):
        row, col = divmod(i, cols)
        try:
            img = PIL.Image.open(img_path).convert("RGB")
            img.thumbnail((THUMB, THUMB), PIL.Image.LANCZOS)
            x = col * (THUMB + 4) + 2
            y = row * (THUMB + 28) + 2
            grid.paste(img, (x, y))
            label = img_path.stem[:25]
            draw.text((x, y + THUMB + 4), label, fill=(180, 180, 200))
            draw.text((x, y + THUMB + 16), f"{img.size[0]}x{img.size[1]}", fill=(120, 120, 140))
        except Exception as e:
            draw.text((col*(THUMB+4)+2, row*(THUMB+28)+2), f"Erro: {e}", fill=(255,80,80))
    
    # Escalar para visualização se muito grande
    max_display = 1600
    if grid.width > max_display:
        ratio = max_display / grid.width
        grid = grid.resize((max_display, int(grid.height * ratio)), PIL.Image.LANCZOS)
    
    display(grid)
    print(f"\n📋 Lista de arquivos:")
    for img_path in imagens:
        size_mb = img_path.stat().st_size / 1024 / 1024
        print(f"   {img_path.name} ({size_mb:.1f} MB)")


preview_geradas(OUTPUT_DIR)

## 💾 Célula 9 — Download das Imagens Geradas

In [ ]:
# ============================================================
# ✏️ CONFIGURAÇÃO DO DOWNLOAD
# ============================================================

# 'zip'   — compactar tudo e baixar um .zip
# 'drive' — copiar para Google Drive (requer montar_drive() antes)
# 'todos' — baixar arquivos individualmente (somente Colab)
MODO_DOWNLOAD = "zip"

# Pasta no Drive para salvar (se MODO_DOWNLOAD = 'drive')
PASTA_DRIVE = "/content/drive/MyDrive/clip-flow/output/mago_gerado"

# ============================================================

imagens_geradas = sorted(
    list(OUTPUT_DIR.glob("*.png")) + list(OUTPUT_DIR.glob("*.jpg"))
)

if not imagens_geradas:
    print("📂 Nenhuma imagem gerada ainda.")
else:
    print(f"📦 {len(imagens_geradas)} imagens para download.")
    
    if MODO_DOWNLOAD == "zip":
        ts = datetime.now().strftime("%Y%m%d_%H%M")
        zip_path = Path(f"mago_mestre_gerado_{ts}.zip")
        
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for img_path in imagens_geradas:
                zf.write(img_path, img_path.name)
                print(f"   + {img_path.name}")
        
        size_mb = zip_path.stat().st_size / 1024 / 1024
        print(f"\n✅ ZIP criado: {zip_path} ({size_mb:.1f} MB)")
        
        if IS_COLAB:
            files.download(str(zip_path))
            print("📥 Download iniciado automaticamente")
        else:
            print(f"📁 Arquivo disponível em: {zip_path.absolute()}")
    
    elif MODO_DOWNLOAD == "drive":
        pasta = Path(PASTA_DRIVE)
        pasta.mkdir(parents=True, exist_ok=True)
        import shutil
        for img_path in imagens_geradas:
            destino = pasta / img_path.name
            shutil.copy2(img_path, destino)
            print(f"   ✓ {img_path.name} → {destino}")
        print(f"\n✅ {len(imagens_geradas)} imagens copiadas para o Drive")
    
    elif MODO_DOWNLOAD == "todos" and IS_COLAB:
        for img_path in imagens_geradas:
            files.download(str(img_path))
            print(f"   📥 {img_path.name}")
        print("\n✅ Downloads iniciados")
    
    else:
        print(f"⚠️ Modo '{MODO_DOWNLOAD}' não disponível neste ambiente.")

## 🔄 Célula 10 — Copiar para assets/backgrounds (uso no pipeline)

Copia as imagens aprovadas diretamente para `assets/backgrounds/mago/` para uso no pipeline Clip-Flow.

In [ ]:
import shutil

# ============================================================
# ✏️ CONFIGURE OS ARQUIVOS A COPIAR
# ============================================================

# Destino no projeto (ajuste se necessário)
DESTINO_ASSETS = Path("assets/backgrounds/mago")

# None = copiar TODAS as imagens geradas
# Lista de nomes = copiar apenas esses
ARQUIVOS_APROVADOS = None  # ou: ["mago_sabedoria_123456.png", ...]

# Prefixo para renomear (None = manter nome original)
PREFIXO_NOVO = "gerado"  # ex: "gerado_sabedoria_xxx.png"

# ============================================================

if not DESTINO_ASSETS.exists():
    print(f"⚠️ Destino não encontrado: {DESTINO_ASSETS}")
    print("   Ajuste DESTINO_ASSETS para o caminho correto do projeto.")
else:
    if ARQUIVOS_APROVADOS:
        fontes = [OUTPUT_DIR / nome for nome in ARQUIVOS_APROVADOS]
    else:
        fontes = sorted(list(OUTPUT_DIR.glob("*.png")) + list(OUTPUT_DIR.glob("*.jpg")))
    
    if not fontes:
        print("📂 Nenhuma imagem para copiar.")
    else:
        copiadas = 0
        for src in fontes:
            if not src.exists():
                print(f"⚠️ Não encontrado: {src.name}")
                continue
            
            if PREFIXO_NOVO:
                novo_nome = f"{PREFIXO_NOVO}_{src.name}"
            else:
                novo_nome = src.name
            
            dst = DESTINO_ASSETS / novo_nome
            shutil.copy2(src, dst)
            print(f"   ✓ {src.name} → {dst}")
            copiadas += 1
        
        print(f"\n✅ {copiadas} imagens copiadas para {DESTINO_ASSETS}")
        print(f"   Agora disponíveis para o pipeline: python -m src.pipeline_cli --mode once")

## 🛠️ Célula 11 — Utilitários e Diagnóstico

In [ ]:
# ============================================================
# Listar modelos disponíveis na API
# ============================================================

def listar_modelos_imagem():
    """Lista os modelos Gemini disponíveis com suporte a geração de imagens."""
    if not GOOGLE_API_KEY:
        print("❌ API Key não configurada")
        return
    
    print("📋 Modelos com 'image' ou 'imagen' no nome:")
    try:
        encontrados = []
        for model in cliente.models.list():
            nome = model.name
            eh_imagem = "image" in nome.lower() or "imagen" in nome.lower()
            if eh_imagem:
                encontrados.append(nome)
                print(f"   ✓ {nome}")
        if not encontrados:
            print("   (nenhum detectado — listando todos:)")
            for model in cliente.models.list():
                print(f"   • {model.name}")
    except Exception as e:
        print(f"⚠️ Erro ao listar modelos: {e}")


def testar_api():
    """Testa conectividade com a API Gemini."""
    if not GOOGLE_API_KEY:
        print("❌ API Key não configurada")
        return False
    
    print("🔍 Testando API...")
    try:
        response = cliente.models.generate_content(
            model="gemini-2.0-flash-lite",
            contents="Responda apenas: OK"
        )
        resposta = response.text.strip() if response.text else "(sem texto)"
        print(f"   ✅ API funcionando. Resposta: {resposta}")
        return True
    except Exception as e:
        print(f"   ❌ Erro: {e}")
        return False


def resumo_estado():
    """Exibe resumo completo do estado atual."""
    print("📊 ESTADO ATUAL DO NOTEBOOK")
    print("=" * 40)
    print(f"🔑 API Key         : {'✓ configurada' if GOOGLE_API_KEY else '✗ ausente — configure antes de continuar'}")
    print(f"📁 Referências     : {len(REFERENCIAS)} imagens carregadas")
    print(f"📂 Saída           : {OUTPUT_DIR}/")

    geradas = list(OUTPUT_DIR.glob("*.png")) + list(OUTPUT_DIR.glob("*.jpg"))
    total_mb = sum(p.stat().st_size for p in geradas) / 1024 / 1024
    print(f"🖼️  Geradas          : {len(geradas)} imagens ({total_mb:.1f} MB)")
    print(f"🌡️  Temperatura      : {TEMPERATURA}")
    print(f"🔗 Refs/geração    : {N_REFERENCIAS}")
    print(f"🤖 Modelos          :")
    for m in MODELOS_IMAGEM:
        print(f"     • {m}")


# Executar diagnóstico
resumo_estado()
print()
testar_api()
print()
listar_modelos_imagem()
